# MNIST handwritten digits (70,000 × 784)

Each row is a 28 × 28 grayscale image of a handwritten digit, flattened to 784 pixel columns (0–255). UMAP should separate the ten digits without ever seeing the labels.

In [ ]:
import gzip
import urllib.request
from pathlib import Path

import numpy as np

CACHE = Path.home() / ".cache" / "positron-atlas" / "datasets"


def download(url: str, name: str) -> Path:
    """Download a file once and cache it."""
    path = CACHE / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"Downloading {url} ...")
        tmp = path.with_suffix(path.suffix + ".part")
        urllib.request.urlretrieve(url, tmp)
        tmp.rename(path)
    return path


def read_idx(path: Path) -> np.ndarray:
    """Read an IDX file (the MNIST format): images as (n, 784) uint8, labels as (n,)."""
    with gzip.open(path, "rb") as f:
        data = f.read()
    ndim = data[3]
    shape = tuple(int.from_bytes(data[4 + 4 * i : 8 + 4 * i], "big") for i in range(ndim))
    arr = np.frombuffer(data, dtype=np.uint8, offset=4 + 4 * ndim)
    return arr.reshape(shape[0], -1) if ndim > 1 else arr

In [ ]:
BASE = "https://ossci-datasets.s3.amazonaws.com/mnist/"
LABELS = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]

files = {
    name: download(BASE + name, "mnist-" + name)
    for name in [
        "train-images-idx3-ubyte.gz",
        "train-labels-idx1-ubyte.gz",
        "t10k-images-idx3-ubyte.gz",
        "t10k-labels-idx1-ubyte.gz",
    ]
}
train_x = read_idx(files["train-images-idx3-ubyte.gz"])
test_x = read_idx(files["t10k-images-idx3-ubyte.gz"])
train_y = read_idx(files["train-labels-idx1-ubyte.gz"])
test_y = read_idx(files["t10k-labels-idx1-ubyte.gz"])
n_train = len(train_y)

pixels = np.concatenate([train_x, test_x])  # (70000, 784) uint8
labels = np.concatenate([train_y, test_y])
pixels.shape, labels.shape

Build a polars DataFrame with one column per pixel (784 numeric columns) and a few columns to color by.

In [ ]:
import polars as pl

mnist = pl.from_numpy(pixels, schema=[f"px{i}" for i in range(pixels.shape[1])]).with_columns(
    pl.Series("label", [LABELS[v] for v in labels], dtype=pl.Enum(LABELS)),
    pl.Series("split", np.where(np.arange(len(labels)) < n_train, "train", "test")),
    pl.Series("ink", pixels.mean(axis=1)),  # average pixel intensity: a numeric column to color by
)
mnist.shape

## Open it in Atlas

1. In the **Variables** pane, double-click **`mnist`** to open it in the Data Explorer.
2. Click **Open in Atlas**, the first button in the Data Explorer's toolbar (top left).

Or run **Atlas: Open DataFrame…** from the Command Palette and pick `mnist`.

## Recommended settings for image data

With the defaults (all 785 numeric columns, standardized, no PCA), the browser UMAP only gets about 12,700 *anchor* rows. That is the limit of 10 million values ÷ 785 dimensions. The kernel then places the other rows next to their nearest anchors. That is quick, but the map is blurry, because z-scoring near-constant border pixels amplifies noise.

For a much sharper map, in the **Data** section:

1. Turn on **Apply PCA first** with **50** components.
2. Turn off **Standardize columns**. Raw pixel intensities are the usual input for these datasets.
3. Click **Apply**.

UMAP then runs on all 70,000 rows: about 4 s with WebGPU (the default when available). Without a GPU, also lower **Sample rows** to about 30 %.

## Things to try

- **Color by `label`.** This is the default, because it is the first low-cardinality categorical column. With the recommended settings below, the ten digits form separate islands. Look for the 4/7/9 and 3/5/8 neighborhoods.
- **Color by `ink`.** Within each digit, the layout often orders thin to bold strokes.
- **Turn on *Points are images*** in the **Display** section, then hover a point to see the digit itself. Atlas finds the 784 pixel columns (`px0` … `px783`) and shows them as a 28 × 28 image. Tick **Invert colors** for black digits on white.
- **Hover** a point to see its row, including its label and split.
- **Drag *Min distance*** to 0 for tight clusters, or to 0.8 for a smoother map. The change applies live.
- **Set *Neighbors*** to 50 or more for more global structure. This rebuilds the graph.